In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, timedelta
from time import sleep, time as now

# === CONFIG ===

# 📌 Output files
output_csv = r"D:\Android_Mobile_App\AndroidProject_5th\github_android_search_results.csv"
output_ranges_csv = r"D:\Android_Mobile_App\AndroidProject_5th\final_ranges_used.csv"

# 📌 Auth
load_dotenv("All_Tokens.env")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in .env")

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json"
}

# 📌 Date range
start_date = datetime.strptime("2015-01-01", "%Y-%m-%d")
end_date = datetime.strptime("2024-12-31", "%Y-%m-%d")

# 📌 Window sizing
initial_window_days = 30
min_window_days = 1
max_window_days = 90
MAX_RESULTS_PER_QUERY = 1000
TARGET_FILL_RATIO = 0.2  # ~800/1000

# === SAFE RATE LIMIT CHECK ===

def check_rate_limit():
    """Check Search API rate limit and sleep if needed."""
    r = requests.get("https://api.github.com/rate_limit", headers=HEADERS)
    if r.status_code != 200:
        print("⚠️  Could not check rate limit — proceeding cautiously.")
        return
    data = r.json()
    remaining = data['resources']['search']['remaining']
    reset_epoch = data['resources']['search']['reset']
    reset_in = max(0, reset_epoch - now())

    print(f"🔎 Search API remaining: {remaining} requests | resets in {reset_in/60:.1f} min")

    if remaining < 5:
        print(f"⏳ Low quota. Sleeping for {reset_in/60:.1f} min until reset.")
        sleep(reset_in + 5)

# === Count helper ===

def check_count(query):
    check_rate_limit()
    params = {"q": query, "per_page": 1}
    r = requests.get("https://api.github.com/search/repositories",
                     headers=HEADERS, params=params)
    if r.status_code != 200:
        print(f"❌ Count check error: {r.status_code} — {r.text}")
        return -1
    return r.json().get("total_count", 0)

# === Fetch helper ===

def fetch_items(query):
    all_items = []
    per_page = 100
    max_pages = 10

    for page in range(1, max_pages + 1):
        check_rate_limit()
        params = {
            "q": query,
            "per_page": per_page,
            "page": page
        }
        r = requests.get("https://api.github.com/search/repositories",
                         headers=HEADERS, params=params)
        if r.status_code != 200:
            print(f"❌ Fetch error: {r.status_code} — {r.text}")
            break
        data = r.json().get("items", [])
        if not data:
            break
        all_items.extend(data)
        sleep(3)  # 👈 longer sleep to stay well under 30 req/min
    return all_items

# === SMART LOOP ===

all_results = []
final_ranges = []

current_start = start_date
current_window_days = initial_window_days

while current_start < end_date:
    while True:
        current_end = current_start + timedelta(days=current_window_days)
        if current_end > end_date:
            current_end = end_date

        date_range = f"created:{current_start.date()}..{current_end.date()}"
        base_query = f"stars:>0 language:Kotlin {date_range}"

        total_count = check_count(base_query)
        print(f"⏳ Checking {current_start.date()} to {current_end.date()} → {total_count} repos (window {current_window_days} days)")

        if total_count == -1:
            print("⚠️  Skipping window due to error.")
            break

        if total_count >= MAX_RESULTS_PER_QUERY and current_window_days > min_window_days:
            current_window_days = max(min_window_days, current_window_days // 2)
            print(f"🔽 Too many results ({total_count}). Shrinking window to {current_window_days} days.")
        elif total_count < (MAX_RESULTS_PER_QUERY * TARGET_FILL_RATIO) and current_window_days < max_window_days:
            new_window = min(max_window_days, current_window_days * 2)
            print(f"🔼 Low count ({total_count}). Expanding window to {new_window} days.")
            current_window_days = new_window
        else:
            print(f"✅ Good window ({total_count}). Fetching repos...")
            items = fetch_items(base_query)
            all_results.extend(items)
            final_ranges.append({
                "start_date": current_start.date(),
                "end_date": current_end.date(),
                "result_count": total_count,
                "window_days": current_window_days
            })
            sleep(5)  # 👈 pause after each chunk for safety
            break

    # Move to next window
    current_start = current_end + timedelta(days=1)

# === SAVE ===

df = pd.json_normalize(all_results)
os.makedirs(os.path.dirname(output_csv), exist_ok=True)
df.to_csv(output_csv, index=False)
print(f"✅ Repos saved to: {output_csv}")

df_ranges = pd.DataFrame(final_ranges)
df_ranges.to_csv(output_ranges_csv, index=False)
print(f"✅ Final ranges saved to: {output_ranges_csv}")

print(df_ranges)


🔎 Search API remaining: 30 requests | resets in 1.0 min
⏳ Checking 2015-01-01 to 2015-01-31 → 41 repos (window 30 days)
🔼 Low count (41). Expanding window to 60 days.
🔎 Search API remaining: 29 requests | resets in 1.0 min
⏳ Checking 2015-01-01 to 2015-03-02 → 99 repos (window 60 days)
🔼 Low count (99). Expanding window to 90 days.
🔎 Search API remaining: 28 requests | resets in 1.0 min
⏳ Checking 2015-01-01 to 2015-04-01 → 177 repos (window 90 days)
✅ Good window (177). Fetching repos...
🔎 Search API remaining: 27 requests | resets in 1.0 min
🔎 Search API remaining: 26 requests | resets in 0.9 min
🔎 Search API remaining: 25 requests | resets in 0.8 min
🔎 Search API remaining: 24 requests | resets in 0.7 min
⏳ Checking 2015-04-02 to 2015-07-01 → 204 repos (window 90 days)
✅ Good window (204). Fetching repos...
🔎 Search API remaining: 23 requests | resets in 0.7 min
🔎 Search API remaining: 22 requests | resets in 0.6 min
🔎 Search API remaining: 21 requests | resets in 0.5 min
🔎 Search A